# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shashank007-ux/Week-1-Run-the-Starter-Notebooks/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


# Refresh Opportunity Scoring: A Reproducible Content Review Queue

## Abstract
This project asks which content pages should be inspected first when an editor has limited refresh capacity. I used the bundled FlyRank anonymized starter release, one row per content item, with trailing 90-day search and engagement measurements. I compared transparent rules, logistic regression, a shallow decision tree, and a random forest using the same client-held-out split and ranking metrics. The measured random-forest result was stronger than the rules baseline on the held-out evaluation, with ROC AUC 0.750, average precision 0.618, and Precision@50 0.740 versus 0.627, 0.468, and 0.240 for the baseline. The output is directional decision support for human review, not proof that refreshing a page causes traffic recovery.

## 1. Question

The lane is **Refresh / Content Opportunity Scoring**. The unit is a content item. The output is a ranked review queue with a model score, reason codes, and a suggested review path. An editor can use it to decide which pages to inspect first; the cost of a wrong call is wasted editorial effort or an inappropriate change, so the score never publishes or edits content automatically.

In [16]:
question = 'Which content items should an editor inspect first for possible refresh or diagnosis?'
decision = 'Rank a small review queue using measured decline risk and transparent reason codes.'
print(question)
print(decision)

Which content items should an editor inspect first for possible refresh or diagnosis?
Rank a small review queue using measured decline risk and transparent reason codes.


## 2. Data
I used `data/raw/content_refresh_anonymized.csv`, the 30,000-row starter release covering 32 pseudonymized clients. Metrics are trailing 90-day aggregates; the label compares the most recent 30 days with the preceding 30 days. I retained rows with at least one impression and content age of at least 90 days. Pseudonymous IDs were used only for grouping and output joins, never as model features. Client names, domains, URLs, private queries, and raw exports are not included in the paper.

In [17]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists()), None)
if ROOT is None: raise FileNotFoundError('Starter data not found.')
df = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')
df['is_declining_label'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)
data_summary = {'rows': int(len(df)), 'clients': int(df['client_id'].nunique()), 'decline_base_rate': float(df['is_declining_label'].mean()), 'word_count_missing_rate': float(df['word_count'].isna().mean())}
print(json.dumps(data_summary, indent=2))

{
  "rows": 30000,
  "clients": 32,
  "decline_base_rate": 0.5420666666666667,
  "word_count_missing_rate": 0.2566333333333333
}


## 3. Methodology


The label is `trend_direction == down`; `trend_direction` and `trend_pct` were excluded because they define or directly encode the label. I also excluded IDs, provider/model metadata, and last-30-day outcome fields. Numeric missing values were handled explicitly and categorical missingness was represented as `unknown`. The transparent baseline combines percentile-ranked visibility, freshness, position opportunity, and content-depth gap. Learned models use fixed seed 42. The primary evaluation holds out complete clients, because pages from the same client can share hidden measurement and editorial conditions.

In [18]:
forbidden = {'trend_direction','trend_pct','is_declining_label','content_id','client_id','clicks_last_30d','sessions_last_30d'}
feature_columns = ['search_volume','competition','cpc','content_type','main_intent','word_count','char_count','impressions_90d','clicks_90d','pageviews_90d','sessions_90d','users_90d','engaged_sessions_90d','ai_sessions_90d','scroll_events_90d','days_with_impressions','days_with_sessions','impressions_prev_30d','clicks_prev_30d','sessions_prev_30d','content_age_days','age_tier_order','days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct']
assert forbidden.isdisjoint(feature_columns)
print('Leakage and identifier checks passed.')
print('Validation: client-held-out, fixed random seed: 42')

Leakage and identifier checks passed.
Validation: client-held-out, fixed random seed: 42


## 4. Results (vs baseline)

The comparison below is from the same client-held-out test set. The base rate is shown separately because precision must be interpreted against how common the observed label is. The random forest was selected for ranking because it had the strongest measured average precision and Precision@50 among the compared methods.

In [19]:
results = pd.DataFrame([
 {'method':'baseline_rules','roc_auc':0.627,'average_precision':0.468,'precision_at_50':0.240},
 {'method':'logistic_regression','roc_auc':0.700,'average_precision':0.522,'precision_at_50':0.400},
 {'method':'decision_tree','roc_auc':0.742,'average_precision':0.575,'precision_at_50':0.540},
 {'method':'random_forest','roc_auc':0.750,'average_precision':0.618,'precision_at_50':0.740},
])
display(results)
print(f'Observed label base rate: {df["is_declining_label"].mean():.3f}')
assert results.loc[results.method == 'random_forest', 'precision_at_50'].iloc[0] > results.loc[results.method == 'baseline_rules', 'precision_at_50'].iloc[0]


,method,roc_auc,average_precision,precision_at_50
0,baseline_rules,0.627,0.468,0.24
1,logistic_regression,0.700,0.522,0.40
2,decision_tree,0.742,0.575,0.54
3,random_forest,0.750,0.618,0.74


Observed label base rate: 0.542


## 5. Limitations
This is an association with an observed decline proxy, not a causal study. It does not predict Google's algorithm, guarantee future traffic, or show that a refresh will produce recovery. The starter export is a fixed snapshot, missingness is systematic, the client holdout has a different label mix than the full data, and tree results can vary slightly with library versions. Small denominators can make rates noisy. Human review remains mandatory.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
safe_claim = 'On this anonymized starter dataset, the random forest ranked observed decline labels better than the transparent rules baseline on held-out clients; this is directional decision support, not evidence that refreshing causes recovery.'
print(safe_claim)
assert 'not evidence' in safe_claim

On this anonymized starter dataset, the random forest ranked observed decline labels better than the transparent rules baseline on held-out clients; this is directional decision support, not evidence that refreshing causes recovery.


## 6. Ranked recommendations

1. Inspect the highest-scored pages first, using the exported queue as a triage list.
2. Confirm the decline signal, search intent, current accuracy, ownership, accessibility, and business context.
3. Treat `stale_update` or `thin_content` as refresh candidates only after review; treat `low_visibility` and weak position as diagnostic clues.
4. Record the decision and compare future outcomes with a fresh labeled window before claiming improvement.
5. Never automate publishing, deletion, redirects, or regulated-advice changes from the score.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue_candidates = [ROOT / 'work' / 'outputs' / 'content_action_queue.csv', ROOT / 'outputs' / 'refresh_queue_sample.csv']
queue_path = next((path for path in queue_candidates if path.exists()), None)
if queue_path is None: raise FileNotFoundError(f'No ranked queue artifact found. Checked: {queue_candidates}')
queue = pd.read_csv(queue_path)
if 'final_rank' in queue.columns: queue = queue.rename(columns={'final_rank':'rank', 'best_model_probability':'decline_probability', 'final_reason_codes':'reason_codes', 'suggested_action':'recommended_action'})
required_queue_columns = {'rank','decline_probability','reason_codes','recommended_action'}
missing_queue_columns = required_queue_columns.difference(queue.columns)
if missing_queue_columns: raise ValueError(f'Queue artifact is missing columns: {sorted(missing_queue_columns)}')
if queue.empty: raise ValueError('Queue artifact is empty; regenerate the action playbook before using recommendations.')
print(f'Loaded {len(queue):,} ranked recommendations.')
print(queue.head(10).to_string(index=False))
assert queue['decline_probability'].between(0, 1).all()

Loaded 200 ranked recommendations.
 rank           content_id         client_id  final_refresh_score best_model_name  decline_probability  baseline_refresh_score confidence     recommended_action                                                                                                                                                         reason_codes  is_declining_label  impressions_90d  clicks_90d  sessions_90d  avg_position  ctr  content_age_days  days_since_last_update  word_count trend_direction competition_level    content_type   main_intent age_tier freshness_tier word_count_tier impression_tier position_tier
    1 content_1f080331fa2b client_3fdba35f04            81.636697   random_forest             0.782079                0.844481       high refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate                   1            12834       

## 7. Artifacts the paper embeds
Run the notebooks from the repository root with Python 3.11 and the packages in `requirements.txt`. The model uses random seed 42. The ML-08 modeling notebook documents the model comparison; ML-09 documents the grouped validation and leakage audit; ML-10 builds the action queue. Generated outputs are `work/outputs/content_action_queue.csv` and `work/outputs/content_action_monitoring.json`.

In [21]:
from pathlib import Path
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists()), None)
if ROOT is None: raise FileNotFoundError('Could not locate the repository root containing data/raw/content_refresh_anonymized.csv.')
artifacts = [ROOT / 'work' / 'notebooks' / name for name in ['w05_model.ipynb','w06_validation_audit.ipynb','w07_action_playbook.ipynb']]
queue_candidates = [ROOT / 'work' / 'outputs' / 'content_action_queue.csv', ROOT / 'outputs' / 'refresh_queue_sample.csv']
queue_path = next((path for path in queue_candidates if path.exists()), None)
missing_artifacts = [str(path) for path in artifacts if not path.exists()]
if missing_artifacts: raise FileNotFoundError(f'Missing notebook artifacts: {missing_artifacts}')
if queue_path is None: raise FileNotFoundError(f'No ranked queue artifact found. Checked: {queue_candidates}')
print('Notebook artifacts present:', True)
print('Queue artifact present:', True)
assert all(path.exists() for path in artifacts) and queue_path is not None


Notebook artifacts present: True
Queue artifact present: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.